
# El módulo `timeit` aplicado a AppTickets

En el cuaderno anterior usamos `time.perf_counter()` para medir manualmente tiempos de ejecución.

Ahora trabajaremos con el módulo integrado `timeit`, que permite hacer mediciones más controladas y repetibles.

Los dos métodos principales que usaremos son:

1. `timeit.timeit()`
2. `timeit.repeat()`

La idea central es:

> `perf_counter()` permite medir manualmente.  
> `timeit` automatiza la medición y facilita repetir pruebas.



## 1. Importar módulos necesarios


In [ ]:

from abc import ABC, abstractmethod
from functools import wraps
from datetime import datetime
import re
import timeit
import os
import pandas as pd



## 2. Aplicación base: AppTickets


In [ ]:

def registrar_accion(funcion):
    @wraps(funcion)
    def envoltura(*args, **kwargs):
        objeto = args[0]

        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        clase = objeto.__class__.__name__
        metodo = funcion.__name__
        folio = getattr(objeto, "folio", "N/A")
        estado_anterior = getattr(objeto, "estado", "N/A")

        resultado = funcion(*args, **kwargs)

        estado_nuevo = getattr(objeto, "estado", "N/A")

        registro = {
            "fecha_hora": fecha_hora,
            "clase": clase,
            "folio": folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }

        if hasattr(objeto, "_historial"):
            objeto._historial.append(registro)

        with open("registro_acciones.log", "a", encoding="utf-8") as archivo:
            archivo.write(
                f"{fecha_hora} | "
                f"{clase} | "
                f"{folio} | "
                f"{metodo} | "
                f"{estado_anterior} -> {estado_nuevo}\n"
            )

        return resultado

    return envoltura


class TicketBase(ABC):

    @abstractmethod
    def atender(self):
        pass


class Ticket(TicketBase):

    ESTADOS_VALIDOS = ["Abierto", "En proceso", "Cerrado"]
    _contador_global = 0

    def __init__(self, folio, titulo, descripcion):
        self._historial = []

        self.folio = folio
        self.titulo = titulo
        self.descripcion = descripcion
        self.estado = "Abierto"

        Ticket._contador_global += 1

        self._registrar_evento_manual("crear", "N/A", self.estado)

    @property
    def folio(self):
        return self._folio

    @folio.setter
    def folio(self, nuevo_folio):
        if not self.es_folio_valido(nuevo_folio):
            raise ValueError(f"Folio no válido: {nuevo_folio}")
        self._folio = nuevo_folio

    @property
    def titulo(self):
        return self._titulo

    @titulo.setter
    def titulo(self, nuevo_titulo):
        self._titulo = nuevo_titulo

    @property
    def descripcion(self):
        return self._descripcion

    @descripcion.setter
    def descripcion(self, nueva_descripcion):
        self._descripcion = nueva_descripcion

    @property
    def estado(self):
        return self._estado

    @estado.setter
    def estado(self, nuevo_estado):
        if nuevo_estado not in self.ESTADOS_VALIDOS:
            raise ValueError(f"Estado no válido: {nuevo_estado}")
        self._estado = nuevo_estado

    @property
    def historial(self):
        return tuple(self._historial)

    def _registrar_evento_manual(self, metodo, estado_anterior, estado_nuevo):
        self._historial.append({
            "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "clase": self.__class__.__name__,
            "folio": self.folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        })

    @registrar_accion
    def cambiar_estado(self, nuevo_estado):
        self.estado = nuevo_estado

    @registrar_accion
    def cerrar(self):
        self.estado = "Cerrado"

    @classmethod
    def total_tickets_creados(cls):
        return cls._contador_global

    @staticmethod
    def es_folio_valido(folio):
        return bool(re.fullmatch(r"^TK-?\d{3}$", folio))

    def registrar_atencion(self):
        self.estado = "En proceso"
        return f"Ticket {self.folio} marcado como En proceso."

    @registrar_accion
    def atender(self):
        return self.registrar_atencion()

    def __str__(self):
        return (
            f"{self.__class__.__name__} | "
            f"Folio: {self.folio} | "
            f"Título: {self.titulo} | "
            f"Estado: {self.estado}"
        )


class TicketSoporte(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de soporte {self.folio} está siendo atendido por mesa de ayuda."
        )


class TicketDesarrollo(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de desarrollo {self.folio} fue asignado al equipo de programación."
        )


class TicketInfraestructura(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de infraestructura {self.folio} fue enviado al área de servidores/redes."
        )



## 3. ¿Qué es `timeit`?

Python incluye un módulo integrado llamado `timeit`.

Este módulo realiza automáticamente muchas de las mediciones que antes hacíamos manualmente con `time.perf_counter()`.

Los métodos más usados son:

| Método | Uso principal |
|---|---|
| `timeit.timeit()` | Ejecuta una instrucción varias veces y devuelve el tiempo total |
| `timeit.repeat()` | Repite varias mediciones completas y devuelve una lista de tiempos |

`timeit` es útil cuando queremos comparar pequeñas piezas de código o repetir una prueba varias veces.



## 4. Parámetros principales de `timeit.timeit()`

El método `timeit.timeit()` puede recibir varios parámetros, pero los más importantes son:

| Parámetro | Significado |
|---|---|
| `stmt` | Instrucción que se va a medir |
| `setup` | Código que se ejecuta antes de iniciar la medición |
| `number` | Número de veces que se ejecutará `stmt` |
| `globals` | Permite usar funciones, clases y variables ya definidas en el notebook |

El tiempo de `setup` no se incluye en el resultado.



## 5. Primer ejemplo: medir validación de folio

Mediremos la validación de un folio.

Como validar un folio es muy rápido, lo ejecutaremos muchas veces para que el tiempo sea visible.


In [ ]:

tiempo_validacion = timeit.timeit(
    stmt='Ticket.es_folio_valido("TK-123")',
    number=100000,
    globals=globals()
)

print(f"Tiempo total: {tiempo_validacion:.8f} segundos")
print(f"Promedio por ejecución: {tiempo_validacion / 10000:.10f} segundos")



## 6. Usar `setup`

El parámetro `setup` sirve para preparar el escenario antes de medir.

En este ejemplo, el ticket se crea antes de la medición.  
Después solo medimos la consulta del estado.

El tiempo de creación del ticket no forma parte del resultado.


In [ ]:

tiempo_consulta_estado = timeit.timeit(
    stmt='ticket.estado',
    setup='ticket = TicketSoporte("TK-101", "Consulta", "Prueba de consulta de estado")',
    number=10000,
    globals=globals()
)

print(f"Tiempo total consultando estado 10,000 veces: {tiempo_consulta_estado:.8f} segundos")
print(f"Promedio por consulta: {tiempo_consulta_estado / 10000:.10f} segundos")



## 7. Reestructurar código para probarlo con `timeit`

Con frecuencia, se necesita reestructurar un fragmento de código para probarlo mejor con `timeit`.

Una forma común es crear funciones temporales.

Esto permite que `stmt` sea más simple y fácil de leer.


In [ ]:

def crear_ticket_temporal():
    return TicketSoporte(
        "TK-201",
        "Ticket temporal",
        "Ticket creado para medir rendimiento."
    )


def validar_folio_temporal():
    return Ticket.es_folio_valido("TK-202")


def cambiar_estado_temporal():
    ticket = TicketSoporte(
        "TK-203",
        "Cambio de estado",
        "Ticket creado para medir cambio de estado."
    )

    ticket.estado = "En proceso"

    return ticket


In [ ]:

tiempo_crear = timeit.timeit(
    stmt='crear_ticket_temporal()',
    number=500,
    globals=globals()
)

tiempo_validar = timeit.timeit(
    stmt='validar_folio_temporal()',
    number=500,
    globals=globals()
)

tiempo_cambiar_estado = timeit.timeit(
    stmt='cambiar_estado_temporal()',
    number=500,
    globals=globals()
)

print(f"Crear ticket 500 veces:      {tiempo_crear:.8f} segundos")
print(f"Validar folio 500 veces:    {tiempo_validar:.8f} segundos")
print(f"Cambiar estado 500 veces:   {tiempo_cambiar_estado:.8f} segundos")



## 8. Presentar resultados en tabla

La tabla permite comparar con mayor claridad los tiempos obtenidos.


In [ ]:

df_timeit = pd.DataFrame([
    {
        "Operación": "Crear ticket",
        "Ejecuciones": 500,
        "Tiempo total (s)": tiempo_crear,
        "Promedio por ejecución (s)": tiempo_crear / 500
    },
    {
        "Operación": "Validar folio",
        "Ejecuciones": 500,
        "Tiempo total (s)": tiempo_validar,
        "Promedio por ejecución (s)": tiempo_validar / 500
    },
    {
        "Operación": "Crear ticket y cambiar estado",
        "Ejecuciones": 500,
        "Tiempo total (s)": tiempo_cambiar_estado,
        "Promedio por ejecución (s)": tiempo_cambiar_estado / 500
    }
])

df_timeit



## 9. Comparar cambio directo contra método decorado

Ahora compararemos:

1. Cambiar el estado directamente.
2. Atender el ticket con el método `atender()`.

Aunque ambas operaciones cambian el estado, no hacen lo mismo.

`ticket.estado = "En proceso"` solo cambia una propiedad.

`ticket.atender()` ejecuta lógica adicional:

- cambia el estado;
- registra historial;
- ejecuta decoradores;
- escribe en archivo log;
- usa comportamiento especializado en las clases hijas.

Por eso, normalmente `atender()` tarda más.


In [ ]:

def limpiar_archivo_log():
    if os.path.exists("registro_acciones.log"):
        os.remove("registro_acciones.log")


def cambiar_estado_directo_temporal():
    ticket = TicketSoporte(
        "TK-301",
        "Cambio directo",
        "Ticket para medir cambio directo."
    )

    ticket.estado = "En proceso"

    return ticket


def atender_ticket_temporal():
    ticket = TicketSoporte(
        "TK-302",
        "Atención decorada",
        "Ticket para medir método atender."
    )

    ticket.atender()

    return ticket


In [ ]:

limpiar_archivo_log()

tiempo_estado_directo = timeit.timeit(
    stmt='cambiar_estado_directo_temporal()',
    number=50,
    globals=globals()
)

limpiar_archivo_log()

tiempo_atender = timeit.timeit(
    stmt='atender_ticket_temporal()',
    number=50,
    globals=globals()
)

df_comparacion = pd.DataFrame([
    {
        "Operación": "Crear ticket y cambiar estado directo",
        "Ejecuciones": 50,
        "Tiempo total (s)": tiempo_estado_directo,
        "Promedio por ejecución (s)": tiempo_estado_directo / 50
    },
    {
        "Operación": "Crear ticket y atender con método decorado",
        "Ejecuciones": 50,
        "Tiempo total (s)": tiempo_atender,
        "Promedio por ejecución (s)": tiempo_atender / 50
    }
])

df_comparacion



## 10. Usar `timeit.repeat()`

`timeit.timeit()` devuelve un solo resultado.

`timeit.repeat()` repite la medición varias veces y devuelve una lista de tiempos.

Parámetros principales:

| Parámetro | Significado |
|---|---|
| `stmt` | Instrucción a medir |
| `setup` | Código previo a la medición |
| `number` | Número de ejecuciones por medición |
| `repeat` | Número de mediciones completas |

Esto permite observar variaciones entre una ejecución y otra.


In [ ]:

repeticiones_validacion = timeit.repeat(
    stmt='Ticket.es_folio_valido("TK-123")',
    number=5000,
    repeat=5,
    globals=globals()
)

repeticiones_validacion


In [ ]:

df_repeticiones = pd.DataFrame({
    "Repetición": range(1, len(repeticiones_validacion) + 1),
    "Tiempo total (s)": repeticiones_validacion,
    "Promedio por ejecución (s)": [valor / 5000 for valor in repeticiones_validacion]
})

df_repeticiones


In [ ]:

print(f"Mejor tiempo:     {min(repeticiones_validacion):.8f} segundos")
print(f"Peor tiempo:      {max(repeticiones_validacion):.8f} segundos")
print(f"Promedio general: {sum(repeticiones_validacion) / len(repeticiones_validacion):.8f} segundos")



## 11. ¿Por qué revisar el menor tiempo?

En pruebas con `repeat()`, muchas veces se revisa el menor tiempo porque representa la ejecución menos afectada por factores externos.

Los tiempos más altos pueden verse afectados por:

- carga del sistema operativo;
- otros procesos abiertos;
- uso de disco;
- interrupciones temporales del entorno;
- carga del propio Jupyter Notebook.

Aun así, también es útil revisar promedio, mínimo y máximo.



## 12. Ejemplo más realista: crear varios tickets

Ahora mediremos una función que crea varios tickets.



In [ ]:

def crear_999_tickets():
    tickets = []

    for numero in range(1, 1000):
        folio = f"TK-{numero:03d}"

        if numero % 3 == 1:
            ticket = TicketSoporte(folio, "Soporte", "Ticket de soporte.")
        elif numero % 3 == 2:
            ticket = TicketDesarrollo(folio, "Desarrollo", "Ticket de desarrollo.")
        else:
            ticket = TicketInfraestructura(folio, "Infraestructura", "Ticket de infraestructura.")

        tickets.append(ticket)

    return tickets


repeticiones_creacion = timeit.repeat(
    stmt='crear_999_tickets()',
    number=5,
    repeat=5,
    globals=globals()
)

df_creacion = pd.DataFrame({
    "Repetición": range(1, len(repeticiones_creacion) + 1),
    "Tiempo total creando 999 tickets x 5 veces (s)": repeticiones_creacion,
    "Promedio por bloque de 999 tickets (s)": [valor / 5 for valor in repeticiones_creacion]
})

df_creacion



## 13. Comparar `timeit.timeit()` contra `timeit.repeat()`

| Método | ¿Qué devuelve? | Uso recomendado |
|---|---|---|
| `timeit.timeit()` | Un solo tiempo total | Cuando necesitas una medición rápida |
| `timeit.repeat()` | Una lista de tiempos | Cuando quieres observar variación |

En palabras sencillas:

- `timeit.timeit()` pregunta: “¿cuánto tardó esto?”
- `timeit.repeat()` pregunta: “¿cuánto tarda esto si lo pruebo varias veces?”


In [ ]:

resumen_timeit = pd.DataFrame([
    {
        "Prueba": "Validar folio con timeit",
        "Método": "timeit.timeit()",
        "Ejecuciones": 10000,
        "Resultado principal (s)": tiempo_validacion
    },
    {
        "Prueba": "Validar folio con repeat",
        "Método": "timeit.repeat()",
        "Ejecuciones": "5000 x 5 repeticiones",
        "Resultado principal (s)": min(repeticiones_validacion)
    },
    {
        "Prueba": "Crear 30 tickets",
        "Método": "timeit.repeat()",
        "Ejecuciones": "5 bloques x 5 repeticiones",
        "Resultado principal (s)": min(repeticiones_creacion)
    }
])

resumen_timeit



## 14. Conclusión didáctica

`timeit` es útil cuando queremos medir código de forma repetible y ordenada.

Ideas clave:

1. `stmt` es lo que se mide.
2. `setup` prepara el entorno antes de medir.
3. `number` controla cuántas veces se ejecuta la instrucción.
4. `timeit.timeit()` devuelve un solo resultado.
5. `timeit.repeat()` devuelve varios resultados.
6. Muchas veces conviene crear funciones temporales para medir mejor.
7. Si una función escribe en archivo, esa escritura también forma parte del tiempo medido.


> `timeit` no adivina qué quieres medir; mide exactamente lo que colocas en `stmt`.
